# MNIST Classification Final Run

Runs the selected final hyperparameters, clean train/test evaluation, and frozen-checkpoint diagnostics.

In [ ]:
from pathlib import Path
import os
import shutil
import sys


def _running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _looks_like_project_root(path: Path) -> bool:
    return (path / "learning_rules_MLP.py").is_file() and (path / "experiment_utils").is_dir()


def _find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for env_name in ["PROJECT_ROOT", "COLAB_PROJECT_ROOT"]:
        value = os.environ.get(env_name)
        if value:
            candidates.append(Path(value))
    candidates.extend([
        Path("/content/backprop-alternatives"),
        Path("/content/drive/MyDrive/backprop-alternatives"),
        Path("/content/drive/MyDrive/colab-folder"),
        Path("/content/drive/MyDrive/Colab Notebooks/drive-folder"),
    ])
    for start in candidates:
        try:
            resolved = start.expanduser().resolve()
        except Exception:
            continue
        for candidate in [resolved, *resolved.parents]:
            if _looks_like_project_root(candidate):
                return candidate
    if _running_in_colab():
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        for hit in Path("/content/drive/MyDrive").rglob("learning_rules_MLP.py"):
            candidate = hit.parent
            if _looks_like_project_root(candidate):
                return candidate
    raise FileNotFoundError("Could not find a folder containing learning_rules_MLP.py and experiment_utils/.")


def _stage_code_locally_if_colab(source_root: Path) -> Path:
    """Import code from /content in Colab instead of reading Python modules from Drive."""
    if not _running_in_colab() or not str(source_root).startswith("/content/drive/"):
        return source_root

    runtime_root = Path("/content/backprop-alternatives-runtime")
    if runtime_root.exists():
        shutil.rmtree(runtime_root)
    runtime_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_root / "learning_rules_MLP.py", runtime_root / "learning_rules_MLP.py")
    shutil.copytree(
        source_root / "experiment_utils",
        runtime_root / "experiment_utils",
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc"),
    )
    (runtime_root / "notebooks").mkdir(exist_ok=True)
    return runtime_root


SOURCE_PROJECT_ROOT = _find_project_root()
PROJECT_ROOT = _stage_code_locally_if_colab(SOURCE_PROJECT_ROOT)
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from experiment_utils.runtime import get_device, setup_matplotlib

setup_matplotlib()
DEVICE = get_device()
print(f"Source project root: {SOURCE_PROJECT_ROOT}")
print(f"Runtime project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Device: {DEVICE}")


In [ ]:
NOTEBOOK_NAME = "mnist-final-run"

TASK_CONFIG = {
    "task_key": "mnist",
    "display_name": "MNIST classification",
    "task_type": "classification",
    "data_loader": "load_mnist",
    "activation": "relu",
    "dimensions": (28 * 28, 256, 128, 10),
    "methods": ["bp", "np", "np_fixed", "np_fan_in", "wp"],
    "seeds": list(range(5)),
    "data_kwargs": {
        "train_limit": None,
        "test_limit": None,
        "train_eval_limit": 4000,
        "batch_size": 128,
        "eval_batch_size": 1024,
        "data_dir": str(DATA_DIR / "torchvision"),
        "seed": 0,
        "mean_center_only": True,
        "num_workers": 0,
    },
    "run_epochs": 400,
    "run_print_every_epoch": 25,
    "run_configs": {
        "bp": {"lr": 0.500},
        "np": {"lr": 0.0510, "sigma": 0.0325},
        "np_fan_in": {"lr": 0.0122, "sigma": 0.00875},
        "np_fixed": {"lr": 0.00563, "sigma": 0.120},
        "wp": {"lr": 0.00375, "sigma": 0.0275},
    },
    "analysis": {
        "epochs": 20,
        "bp_lr": 0.200,
        "num_perturbations": 100,
        "batch_size": 128,
        "checkpoint_epochs": [1, 10, 20],
        "method_sigmas": {
            "np": 0.0325,
            "np_fan_in": 0.00875,
            "np_fixed": 0.120,
            "wp": 0.0275,
        },
    },
}


In [ ]:
from experiment_utils.final_runs import run_final_config

outputs = run_final_config(TASK_CONFIG, project_root=PROJECT_ROOT, show=True, device=DEVICE)


In [ ]:
# Optional manual export/download. Set EXPORT_RESULTS=True after the run has completed.
EXPORT_RESULTS = False

if EXPORT_RESULTS:
    from experiment_utils.export import download_if_colab, export_outputs

    archive_path = export_outputs(outputs, archive_name=NOTEBOOK_NAME)
    download_if_colab(archive_path)
